# Phase 4 — Naming clusters with TF-IDF

Each cluster is treated as one document. TF-IDF picks terms that are **common inside the cluster** but
**rare in other clusters**: the top 5 become the cluster's automatic label.

Code: `src/labeling.py`

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../src")
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from labeling import *
pd.set_option("display.max_colwidth", 120)

text = load_job_text()
text_clusters = pd.read_csv(PROCESSED / "text_clusters.csv", index_col="code")
numeric_clusters = pd.read_csv(PROCESSED / "numeric_clusters.csv", index_col="code")
profiles = pd.read_csv(PROCESSED / "numeric_cluster_profiles.csv", index_col=0)["defining_attributes"]

## 1. Why not plain TF-IDF on concatenated text?
The textbook approach joins every task in a cluster into one long document. The problem: a single job with a
long, jargon-heavy task list can dominate. Here's what that gives for the Phase 2 HDBSCAN clusters:

In [2]:
labels = text_clusters["hdbscan"][text_clusters["hdbscan"] != -1]
docs = text.loc[labels.index].groupby(labels).agg(" ".join)
vec = TfidfVectorizer(stop_words=STOP_WORDS, ngram_range=(1, 2), sublinear_tf=True, max_df=0.5)
M = vec.fit_transform(docs)
vocab = vec.get_feature_names_out()
naive = pd.Series({c: [vocab[j] for j in M[i].toarray().ravel().argsort()[::-1][:5]] for i, c in enumerate(docs.index)})
naive.loc[[1, 11, 12, 16]]

1                                     [patients, treatment plans, therapy, prescribe, nurses]
11                                    [remote sensing, sensing, gis, systems gis, wastewater]
12             [identification device, device rfid, frequency identification, rfid, web site]
16    [fuel cell, mems, microelectromechanical, systems mems, microelectromechanical systems]
dtype: object

Cluster 11 (75 jobs) is labeled by *remote sensing / GIS* and cluster 12 by *RFID* — terms from one or two jobs.

**Fix:** define TF as the **share of the cluster's jobs** that use a term, not a word count. A term only scores
high if many jobs in the cluster share it. IDF still penalizes terms used across many clusters.
Near-duplicates (*patient / patients*, or a word already inside a chosen two-word phrase) are skipped.

## 2. Labels for the 31 text clusters (Phase 2, HDBSCAN)

In [3]:
text_labels = top_terms(text, text_clusters["hdbscan"])
text_labels["examples"] = [", ".join(text_clusters.title[text_clusters.hdbscan == c].head(3)) for c in text_labels.index]
text_labels

,size,top_terms,label,examples
cluster,,,,
0,30,"[parents, aids, students progress, visual aids, field trips]",Parents · Aids · Students Progress,"Training and Development Managers, Education and Childcare Administrators, Preschool and Daycare, Education Administ..."
1,118,"[patients, physicians, therapy, treatment, care]",Patients · Physicians · Therapy,"Clinical Research Coordinators, Fitness and Wellness Coordinators, Health Informatics Specialists"
2,21,"[postsecondary, office hours, materials syllabi, participate student, assignments papers]",Postsecondary · Office Hours · Materials Syllabi,"Education Administrators, Postsecondary, Astronomers, Computer Science Teachers, Postsecondary"
3,16,"[student recruitment, postsecondary teach, participate campus, grades records, academic vocational]",Student Recruitment · Postsecondary Teach · Participate Campus,"Business Teachers, Postsecondary, Architecture Teachers, Postsecondary, Area, Ethnic, and Cultural Studies Teachers,..."
4,28,"[menus, kitchen, dishes, meats, serving]",Menus · Kitchen · Dishes,"Food Service Managers, Dietitians and Nutritionists, Dietetic Technicians"
5,18,"[collections, reference, significance, reference materials, sources]",Collections · Reference · Significance,"Document Management Specialists, Anthropologists and Archeologists, Historians"
6,10,"[judges, trial, attorneys, judicial, motions]",Judges · Trial · Attorneys,"Lawyers, Judicial Law Clerks, Administrative Law Judges, Adjudicators, and Hearing Officers"
7,27,"[criminal, surveillance, investigation, law enforcement, suspects]",Criminal · Surveillance · Investigation,"Security Managers, Loss Prevention Managers, Government Property Inspectors and Investigators"
8,17,"[productions, television, music, sound, directors]",Productions · Television · Music,"Actors, Producers and Directors, Media Programming Directors"


In [4]:
compare = pd.DataFrame({"plain TF-IDF": naive, "job-share TF-IDF": text_labels.top_terms})
compare.loc[[1, 11, 12, 16, 17, 26]]

,plain TF-IDF,job-share TF-IDF
1,"[patients, treatment plans, therapy, prescribe, nurses]","[patients, physicians, therapy, treatment, care]"
11,"[remote sensing, sensing, gis, systems gis, wastewater]","[land, soil, hazards, environmental, impacts]"
12,"[identification device, device rfid, frequency identification, rfid, web site]","[hardware software, users, network, server, logical]"
16,"[fuel cell, mems, microelectromechanical, systems mems, microelectromechanical systems]","[engineers, components, engineers design, components systems, engineering]"
17,"[wells, pipe, blast, explosives, sewer]","[levers, trucks, oil gas, drive, pipes]"
26,"[overhaul, test operation, repair overhaul, tires, timepieces]","[wiring, repairers, electrical, replace, reassemble]"


## 3. Names for the 6 broad groups (Phase 3, numeric K=6)

For the top level of the bubbles, auto-generated terms aren't friendly enough for someone browsing careers.
Each group gets a **hand-written name and "I want to…" tagline**, matched by its defining attributes (not its
cluster number, which can change between runs). TF-IDF terms are kept alongside as supporting keywords.

In [5]:
groups = name_groups(profiles).join(top_terms(text, numeric_clusters["kmeans6"])[["size", "top_terms"]])
groups["defining_attributes"] = profiles
groups

,name,tagline,size,top_terms,defining_attributes
0,Tech & Analysis,"I want to solve problems with technology, science and data",137,"[engineering, engineers, project, models, software]","Programming, Mathematical Reasoning, Mathematics, Technology Design, Engineering and Technology"
1,Build & Fix,"I want to build, repair and operate machines and equipment",201,"[hand, hand tools, repair, malfunctions, equipment]","Equipment Maintenance, Repairing, Repairing and Maintaining Mechanical Equipment, Reaction Time, Operation and Control"
2,Teach & Lead,"I want to teach, communicate with and lead people",163,"[evaluate, community, academic, media, courses]","Sociology and Anthropology, Learning Strategies, Speech Clarity, Speaking, Instructing"
3,Help & Heal,I want to help people and care for their health,164,"[patients, equipment, emergency, supervise, evaluate]","Assisting and Caring for Others, Medicine and Dentistry, Psychology, Therapy and Counseling, Biology"
4,Hands-On Work,"I want active, physical, hands-on work",122,"[hand, clean, hand tools, remove, cut]","Trunk Strength, Stamina, Static Strength, Extent Flexibility, Handling and Moving Objects"
5,Organize & Serve,"I want to organize, manage records and serve customers",123,"[questions, mail, contact, payment, answer]","Administrative, Performing Administrative Activities, Customer and Personal Service, Performing for or Working Direc..."


## 4. Findings
- Job-share TF-IDF gives clear, recognizable labels for the fine clusters: *judges / trial / attorneys*,
  *menus / kitchen / dishes*, *needles / fabric / sewing*, *accounts / funds / tax*.
- A few labels are still generic (e.g. *engineers / components*); these will be reviewed in the hierarchy step.
- For broad groups, TF-IDF terms are too mixed to work as names, so plain-language names are used instead.
- `top_terms()` works for any set of cluster labels, so Phase 5 will reuse it for the subclusters inside each group.

In [6]:
text_labels.assign(top_terms=text_labels.top_terms.str.join(", ")).to_csv(PROCESSED / "text_cluster_labels.csv")
groups.assign(top_terms=groups.top_terms.str.join(", ")).to_csv(PROCESSED / "group_names.csv")
print("saved")

saved
